<a href="https://colab.research.google.com/github/nicolaiberk/llm_ws/blob/main/notebooks/05b_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# API Calls

Check out [Inference Endpoints](https://endpoints.huggingface.co).

This time, we do not need a GPU - all computation is running on a server.

In [ ]:
!pip install pydantic_ai "pydantic-ai-slim[huggingface]"

> ❗ RESTART THE NOTEBOOK (DROPDOWN NEXT TO RUN ALL > RESTART SESSION)

In [ ]:
HF_TOKEN = '' # you received this key via email

Making a simple API call

In [ ]:
import requests

def query(payload):
	headers = {
		"Accept" : "application/json",
		"Authorization": f"Bearer " + HF_TOKEN,
		"Content-Type": "application/json"
	}
	response = requests.post(
		# note the endpoint URL: /v1/chat/completions is used for chat completion
		"https://pr04mvi6mfgikrkc.eu-west-1.aws.endpoints.huggingface.cloud/v1/chat/completions",
		headers=headers,
		 json={
        "model": "endpoint",  # ignored, the endpoint serves one model
        "messages": payload,
        "max_tokens": 50,
    	},
	)
	return response.json()

output = query([{
	"role": "user",
	"content": "A small step for a man",
	"parameters": {
		"temperature": 0,
		"max_new_tokens": 50
	}
}])


In [ ]:
print(output["choices"][0]["message"]["content"])

## Getting embeddings via the Inference API

APIs are not just for text generation — we can also request embeddings. Here we use Hugging Face's serverless Inference API with a sentence-transformer model: we send a list of sentences and get back one embedding vector per sentence.

In [ ]:
embedding_url = "https://srrcjsqjtlh4ulxx.eu-west-1.aws.endpoints.huggingface.cloud"

def get_embeddings(texts):
	headers = {"Authorization": f"Bearer {HF_TOKEN}"}
	response = requests.post(
		embedding_url,
		headers=headers,
		json={"inputs": texts}
	)
	return response.json()

sentences = [
	"The government announced a new tax reform.",
	"Parliament passed legislation on fiscal policy.",
	"My cat loves sleeping in the sun."
]

embeddings = get_embeddings(sentences)

print(f"{len(embeddings)} embeddings with {len(embeddings[0])} dimensions each")

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Sentence 1 vs 2:", round(cosine_similarity(embeddings[0], embeddings[1]), 3))
print("Sentence 1 vs 3:", round(cosine_similarity(embeddings[0], embeddings[2]), 3))

## Controlling model output structure with `pydantic`

In [ ]:
## this code is necessary to use pydantic in notebooks, see https://ai.pydantic.dev/troubleshooting/
import nest_asyncio
nest_asyncio.apply()

More on using OpenAI-compatible endpoints with pydantic-ai: https://ai.pydantic.dev/models/openai/#openai-compatible-models

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent, NativeOutput
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

class CityLocation(BaseModel):
    # field descriptions become part of the schema the model sees -
    # use them to steer what exactly goes into each field
    city: str = Field(description="Name of the city , e.g. 'Berlin'")
    country: str = Field(description="Country where the city is located in, e.g. 'Germany'.")

model = OpenAIChatModel(
    'endpoint-model',  # ignored
    provider=OpenAIProvider(
        base_url="https://pr04mvi6mfgikrkc.eu-west-1.aws.endpoints.huggingface.cloud/v1",
        api_key=HF_TOKEN,
    ),
)

# NativeOutput asks the server to constrain generation to our schema
agent = Agent(model, output_type=NativeOutput(CityLocation))

In [ ]:
result = agent.run_sync('Where were the olympics held in 2012?')

In [ ]:
result.output.city

In [ ]:
result.output.country

# Exercise

1. Think of a concept from your own research that you would like to measure in texts (e.g. populism, policy positions, protest frames, emotional tone, ...). Define a `pydantic` `BaseModel` describing the structured annotation you want for each text. Use at least two fields — for example a label and a short justification. *Hint: for a fixed set of categories, you can type a field as `Literal['label_a', 'label_b']` from the `typing` module.*
2. Write three to four short example texts to annotate (or take real ones from your data). Include at least one text you consider ambiguous.
3. Create a `pydantic_ai` `Agent` with your schema as `output_type` (wrapped in `NativeOutput`), reusing the `model` object from above. Write a prompt that briefly explains your concept, then annotate each of your example texts with `run_sync`.
4. Inspect the outputs. Do the annotations match your own judgement? Where does the schema constrain the model in useful ways — and where does it *force* an answer even though the text is ambiguous?

**BONUS:** Add a `confidence` field (a `float` between 0 and 1) to your schema and re-run the annotation. Does the model's reported confidence line up with the texts *you* found ambiguous? What does this tell you about how much to trust such scores?

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# 1. define your schema
class MyFirstSchema(BaseModel):
    ...


In [ ]:
# 2. your example texts
texts = [
    ...
]

In [ ]:
# 3. define your agent and annotate each text


## BONUS: Calling the Anthropic API with a tool

Commercial providers like Anthropic offer their own APIs. A particularly powerful feature are **tools**: capabilities the model can decide to invoke while answering. Here we use a **server-side tool** — web search — which Anthropic executes for us: the model decides *whether* and *what* to search, the provider runs the search, and the results flow back into the model's answer. This way, the model can answer questions about events *after* its training cutoff.

You'll need an API key from [console.anthropic.com](https://console.anthropic.com/).

In [ ]:
!pip install anthropic

In [ ]:
import anthropic

os.environ['ANTHROPIC_API_KEY'] = ''

client = anthropic.Anthropic()

In [ ]:
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=16000,
    tools=[{
        "type": "web_search_20260209",
        "name": "web_search",
        "max_uses": 3  # cap the number of searches per request
    }],
    messages=[{
        "role": "user",
        "content": "Who is the local head of the SPD in Saxony, Germany?"
    }],
)

Note that this will run for a while. Let's look at what the model searched for and print its final answer:

In [ ]:
response.content

In [ ]:
response.content[1].input

In [ ]:
## what did the model do?
for block in response.content:
    if block.type == "server_tool_use":
        print(f"🔍 searched for: {block.input['query']}")
    elif block.type == "web_search_tool_result":
        print(f"   → received {len(block.content)} results")

## the final answer
print()
print("".join(block.text for block in response.content if block.type == "text"))